# 모듈 1: 블랙파우더 성분 데이터 전처리 - EDA 및 검증

`src/pipeline.py`의 각 단계를 순서대로 호출해 원본 데이터를 정제하고, 전/후 비교 시각화와 파생변수·원료등급 기여도(ablation test)를 확인한다.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
from pathlib import Path

from src.preprocessing import load_raw, clean_schema
from src.imputation import flag_outliers, impute_missing
from src.features import engineer_features
from src.validation import (
    check_ranges,
    plot_missing_before_after,
    plot_distributions,
    plot_correlation_heatmap,
    run_ablation_test,
    plot_ablation_results,
    plot_grade_distribution,
)
from src.config import COMPOSITION_COLUMNS, PROCESS_COLUMNS, TARGET_COLUMNS

RAW_PATH = Path('../data/raw/영풍그룹_금속회수율최적화_RawMaterialData_학생배포용.xlsx')
FIGURES_DIR = Path('../reports/figures')

## 1. 원본 로드 및 정제

In [ ]:
raw_df = load_raw(RAW_PATH)
clean_df = clean_schema(raw_df)
outlier_flagged_df = flag_outliers(clean_df)
imputed_df = impute_missing(outlier_flagged_df)
final_df = engineer_features(imputed_df)
final_df.head()

### 1.1 모델링용 중복 배치 제거

`배치ID` 기준 중복 4건(8행)은 분석일자·원료등급·회수율까지 완전히 동일한 복사본으로 확인됐다. 삭제하지 않고 감사 목적상 `is_duplicate_batch` 플래그만 남긴 원본(`final_df`, 424행)은 그대로 보존하되, 교차검증에 동일 데이터가 두 번 들어가면 학습·평가에 같은 포인트가 중복 반영되어 성능이 부풀려질 수 있으므로, 상관관계·ablation test 등 모델링에는 별도로 중복을 제거한 `model_df`를 사용한다.

In [ ]:
model_df = final_df.drop_duplicates(subset='배치ID', keep='first')
print(f'final_df: {len(final_df)}행 / model_df(중복 제거): {len(model_df)}행')

## 2. 결측치 처리 전/후 비교

In [ ]:
plot_missing_before_after(outlier_flagged_df, imputed_df, FIGURES_DIR)

## 3. 이상치 제거 전/후 분포

In [ ]:
plot_distributions(raw_df, ['Ni_함량_pct', '황산투입량_L', '침출온도_C'], FIGURES_DIR / 'before')
plot_distributions(final_df, ['Ni_함량_pct', '황산투입량_L', '침출온도_C'], FIGURES_DIR / 'after')

## 4. 상관관계 히트맵

In [ ]:
feature_columns = COMPOSITION_COLUMNS + PROCESS_COLUMNS + [
    'acid_to_ore_ratio', 'leach_intensity', 'total_impurity_pct', 'target_purity_ratio',
    'Fe_Ni_ratio', 'Cu_Ni_ratio', 'Al_Ni_ratio'
]
plot_correlation_heatmap(model_df, feature_columns, TARGET_COLUMNS, FIGURES_DIR)

조성 변수 중 목표금속(Ni_함량_pct, Co_함량_pct, Li_함량_pct)은 회수율과 양의 상관(+0.30~+0.49), 불순물(Cu/Fe/Al_불순물_pct)과 Mn_함량_pct는 음의 상관(-0.26~-0.57)을 보여 — 원료 안에 목표금속이 많을수록, 불순물이 많을수록 회수율에 미치는 방향이 §6 특성공학에서 가정한 도메인 지식(경쟁 반응 가설)과 일치한다. 공정조건(침출온도_C, 침출시간_hr)도 양의 상관(+0.27~+0.48)을 보여 반응속도론적 기대와 부합한다.

특히 파생변수(acid_to_ore_ratio, leach_intensity, total_impurity_pct, target_purity_ratio)는 대체로 그 구성요소인 원본 변수 개별 상관보다 더 강하다 — 예: leach_intensity(Ni 0.51)는 구성요소인 침출온도_C(0.48)·침출시간_hr(0.36) 개별 상관보다 높고, total_impurity_pct(Co -0.60)도 Cu/Fe/Al_불순물_pct 개별 상관(-0.53~-0.57)보다 더 강하다. 이는 §6 ablation test에서 파생변수가 R²를 크게 끌어올린 결과(Ni 0.55→0.81)와 방향이 일치한다.

반면 Fe_Ni_ratio·Cu_Ni_ratio·Al_Ni_ratio도 -0.41~-0.54 수준의 꽤 강한 단순 상관을 보이는데, §6.1 ablation test에서는 모델 R²에 전혀 기여하지 못했다(§6.1 결론 참고) — 변수 하나와 타겟 사이의 단순 상관과, 다른 변수들과 함께 썼을 때의 모델 기여도는 서로 다른 질문이라는 걸 보여주는 사례다.

## 5. 정합성 검증

In [ ]:
violations = check_ranges(final_df)
print('정상범위 위반:', violations if violations else '없음')

## 6. 파생변수 및 원료등급 기여도 검증 (Ablation Test)

In [ ]:
base_features = COMPOSITION_COLUMNS + PROCESS_COLUMNS
# analysis_month는 회수율 예측 후보 변수가 아니라 계절성/설비 드리프트 모니터링용 캘린더 파생변수이므로, 특성 기여도 비교(히트맵·ablation test) 대상에서 제외한다.
derived_features = ['acid_to_ore_ratio', 'leach_intensity', 'total_impurity_pct', 'target_purity_ratio']

feature_sets = {
    'base_only': base_features,
    'base_plus_grade': base_features + ['원료등급'],
    'base_plus_derived': base_features + derived_features,
    'all_features': base_features + ['원료등급'] + derived_features,
}

results = run_ablation_test(model_df, feature_sets, TARGET_COLUMNS)
pd.DataFrame(results).T

In [ ]:
plot_ablation_results(results, FIGURES_DIR)

### 6.1 개별 불순물 비율(Fe/Ni, Cu/Ni, Al/Ni) 추가 검증

`total_impurity_pct`는 Cu/Fe/Al 불순물을 동일 가중치로 합산해, 어느 불순물이 실제로 회수율 저하를 유발하는지 정보를 가린다. 위 §6 ablation과 동일한 4-케이스 구조로, 비율 변수가 (a) 단독으로도 신호가 있는지, (b) 이미 검증된 최선 조합(`base_plus_derived`) 위에 추가로 기여하는지를 함께 검증한다.

In [ ]:
impurity_ratio_features = ['Fe_Ni_ratio', 'Cu_Ni_ratio', 'Al_Ni_ratio']

ratio_feature_sets = {
    'base_only': base_features,
    'base_plus_ratios': base_features + impurity_ratio_features,
    'base_plus_derived': base_features + derived_features,
    'base_plus_derived_plus_ratios': base_features + derived_features + impurity_ratio_features,
}

ratio_results = run_ablation_test(model_df, ratio_feature_sets, TARGET_COLUMNS)
pd.DataFrame(ratio_results).T

In [ ]:
plot_ablation_results(ratio_results, FIGURES_DIR / 'ratios')

**결론**: 4개 케이스 실측 결과는 다음과 같다 (중복 배치 제거한 `model_df` 기준).

- `base_only` → `base_plus_ratios`: Ni R² 0.5495±0.0457 → 0.5436±0.0490 (차이 -0.0059), Co R² 0.5900±0.0552 → 0.5834±0.0597 (차이 -0.0066) — **비율 변수는 단독으로도 아무 신호가 없다.**
- `base_plus_derived` → `base_plus_derived_plus_ratios`: Ni R² 0.8071±0.0224 → 0.7997±0.0249 (차이 -0.0074), Co R² 0.7927±0.0138 → 0.7924±0.0170 (차이 -0.0003) — 이미 검증된 최선 조합 위에 추가해도 역시 변화 없음.

두 비교 모두 차이가 fold별 표준편차(r2_std, 약 0.01~0.06)보다 훨씬 작아 **통계적으로 노이즈 수준과 구분이 안 되는 차이**다. 상관관계 히트맵에서는 Fe_Ni_ratio·Cu_Ni_ratio·Al_Ni_ratio가 타겟과 -0.4~-0.5 수준의(결코 약하지 않은) 단순 상관을 보였는데도 모델 R²에는 전혀 기여하지 못했다는 점이 흥미롭다 — RandomForest는 이미 `Cu/Fe/Al_불순물_pct`와 `Ni_함량_pct` 원본 컬럼에 직접 접근할 수 있어서, 비율을 미리 계산해주지 않아도 트리 분기로 비슷한 경계를 스스로 학습할 수 있기 때문으로 보인다(이는 비율 항이 명시적으로 필요한 선형모델과의 차이점이다). 따라서 이 세 비율 변수는 `engineer_features`에는 유지하되(해석용 참고 지표로서의 가치는 있을 수 있음), 최종 모델 입력 특성 세트에는 포함하지 않기로 결론짓는다 — "단순 상관계수가 높아 보인다"는 것과 "모델에 실제로 기여한다"는 것은 다른 질문이며, 후자는 반드시 ablation test로 (평균뿐 아니라 fold 간 변동폭까지 고려해) 검증해야 한다는 것을 보여주는 사례다.

## 7. 심층 인터뷰 답변

**1) 실제 공정 데이터 품질 문제를 어떻게 식별·처리했는가?**

데이터사전에 정의된 정상 범위를 벗어나는 값(`Ni_함량_pct=999.0`, `침출온도_C=-5.0`, `황산투입량_L=15000`)을 물리적으로 불가능한 이상치로 식별해 `flag_outliers`에서 NaN으로 치환했다. 이후 남은 랜덤 결측(조성 6~9%, 공정조건)은 변수 성격에 따라 원료등급별 중앙값, 회귀 기반, 배치ID 순서 기반 보간으로 구분 처리했다 (설계 문서 §4).

**2) 특성공학 과정에서 도메인 지식을 어떻게 활용했는가?**

산-광석비, 침출강도(온도×시간), 총불순물, 목표금속 순도비는 모두 습식제련 침출 공정의 화학양론·반응속도론적 지식에서 도출했다 (설계 문서 §6). 위 ablation test로 파생변수·원료등급 포함 시 R²/RMSE가 실제로 개선되는지 정량 확인했다.

**3) 데이터 불균형 문제가 있었다면 어떻게 해결했는가?**

전체 424행 기준 `원료등급`(A/B/C) 분포는 B 38.9%, A 35.1%, C 25.9%로(아래 셀 실행 결과), 가장 적은 클래스(C)도 약 26%를 차지해 특정 클래스가 극단적으로 희소한 심각한 불균형은 아니라고 판단했다. 설계 문서 §9에서는 구현 단계의 실제 분포를 보고 불균형 여부와 계층적 샘플링(stratified split) 적용 여부를 결정하기로 했는데, 실측 결과 클래스 간 비율 차이가 26~39% 수준으로 완만해 SMOTE·계층적 샘플링·클래스 가중치 등 별도의 불균형 보정은 적용하지 않기로 결론지었다. 다만 위 ablation test 결과(중복 배치 제거한 `model_df` 기준)를 보면 `base_only`(Ni R²=0.5495±0.0457, Co R²=0.5900±0.0552) 대비 `base_plus_grade`(Ni R²=0.5495±0.0450, Co R²=0.5876±0.0530)는 Ni는 거의 그대로, Co는 오히려 소폭 하락(-0.0024)했는데, 이는 fold별 표준편차(r2_std≈0.04~0.06)보다 훨씬 작아 통계적으로 유의미한 변화라고 볼 수 없다 — `원료등급`을 모델 입력 특성으로 그대로 추가하는 것 자체는 회귀 성능에 영향을 주지 않는다. 그럼에도 `원료등급`은 Task 5의 등급별 중앙값 보간(grade-based median imputation)에서 그룹핑 키로 쓰이므로 컬럼 자체는 계속 유지한다. 반면 산-광석비·침출강도·총불순물·목표금속 순도비 등 파생변수는 `base_plus_derived`에서 R²를 Ni 0.5495→0.8071, Co 0.5900→0.7927 수준으로 크게 끌어올려, 현재 데이터에서 회귀 성능에 실질적으로 기여하는 것은 `원료등급`이 아니라 파생변수 쪽임을 ablation test가 보여준다.

In [ ]:
final_df['원료등급'].value_counts(normalize=True) * 100

In [ ]:
plot_grade_distribution(final_df, FIGURES_DIR)